# Calibration: local results against the scoring hardware


In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

In [ ]:
import json, time
T0 = time.time()
CALIB = {'started': time.strftime('%Y-%m-%d %H:%M:%S'), 'models': {}}
CALIB_PATH = Path('/kaggle/working/calibration.json')

def save():
    CALIB['elapsed_seconds'] = round(time.time() - T0, 1)
    CALIB_PATH.write_text(json.dumps(CALIB, indent=2))

save()

In [ ]:
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
tasks = load_tasks(TASKS_PATH)
TASK_IDS = ["fastapi_11355", "fastapi_13920", "fastapi_14262", "fastapi_14349", "fastapi_14372", "fastapi_14463", "fastapi_14482", "fastapi_14485", "fastapi_14492", "fastapi_14583", "fastapi_14609", "fastapi_14873", "fastapi_14986", "fastapi_15023", "fastapi_15280", "fastapi_15588", "fastapi_15589", "fastapi_5624", "rich_3061", "rich_3278", "rich_3454", "rich_3471", "rich_3480", "rich_3777", "rich_3905", "rich_3938", "rich_3944", "rich_3953", "rich_4006", "rich_4077"]
SELECTED = [t for t in tasks if t.instance_id in TASK_IDS]
assert len(SELECTED) == len(TASK_IDS), (len(SELECTED), len(TASK_IDS))

FILES = {
 "agent.yaml": "name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n",
 "prompts/system.md": "You fix one issue in a Python repository under `/workspace`. The problem statement is the title and description of the pull request that fixed it, so it usually says what was changed.\n\nYou have a limited time and tool-call budget (stated in the task). Every tool call costs time, and writing long text costs far more. Work in this order.\n\n## 1. Locate\n- Pull identifiers from the problem statement: function, class, parameter, option name, error text.\n- Find them with ONE `run_command` that chains several searches, for example:\n  `grep -rn \"name_a\" --include=*.py . | grep -v -e tests/ -e docs/ | head -15; grep -rn \"name_b\" --include=*.py . | head -15`\n- Then read only the function you need with `read_file`, passing `start_line` and `end_line` (60 lines or fewer). Do not read whole files.\n\n## 2. Edit (as soon as you know where)\n- Change the source with `edit_file`: a short `old_string` copied exactly from what you read, and its `new_string`.\n- Make the smallest change that does what the problem statement describes.\n- An edit on disk is graded even if time runs out. No edit is a certain failure. If you are torn between two fixes, apply the more likely one now and refine it afterwards.\n\n## 3. Check\n- Run the closest existing test file once: `python -m pytest tests/test_x.py -x -q 2>&1 | tail -15`\n- If it fails because of your change, fix the source. Ignore failures unrelated to your change.\n\n## 4. Submit\n- Call `submit_patch`, then reply with one sentence. `submit_patch` ends the session, so call it only when you are done.\n\n## Rules\n- Think for a sentence or two before each call. Never plan at length.\n- Never create files: no reproduction scripts, no new tests. For a quick check, use `python -c \"...\"` inside `run_command`. Use `write_file` only when the task itself requires a new source file.\n- Never edit tests, `conftest.py`, `pytest.ini`, `pyproject.toml` or `setup.cfg`. Those changes are discarded.\n- Never run the whole test suite and never `pip install`.\n- When the fix is in place, or fewer than 5 tool calls remain, submit.\n",
 "configs/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 2048\nthinking_config:\n  thinking_budget: 512\n  include_thoughts: true\n",
 "eval_config.yaml": "evaluation:\n  timeout_seconds: 120\n  max_tool_calls: 100\n  max_time_minutes: 5\n  max_turns: 250\n"
}


def write_submission(directory: Path, model_name: str) -> Path:
    if directory.exists():
        shutil.rmtree(directory)
    for rel, text in FILES.items():
        path = directory / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        if rel == 'agent.yaml':
            text = text.replace('model: gemma-4-31b-it-qat-w4a16-ct', f'model: {model_name}')
        path.write_text(text, encoding='utf-8')
    return directory


SUBS = {
    '31b': write_submission(WORKING_DIR / 'sub_31b', 'gemma-4-31b-it-qat-w4a16-ct'),
    '12b': write_submission(WORKING_DIR / 'sub_12b', 'gemma-4-12b-it-qat-w4a16-ct'),
}
CALIB['task_ids'] = TASK_IDS
save()
print(len(SELECTED), 'tasks;', {k: (v / 'agent.yaml').read_text().splitlines()[1] for k, v in SUBS.items()})

In [ ]:
import asyncio
import concurrent.futures
import yaml
from adk_eval_core.tracing import SessionTrace
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


class GoldEvaluator(Evaluator):
    """Grades each task's reference patch instead of running the agent (grader control)."""

    async def _run_agent_sandbox(self, task, snapshot_path, *args, **kwargs):
        return task.patch, None, SessionTrace()


def make_eval_config(sub_dir: Path, models, adapters, results_dir: Path) -> EvalConfig:
    ev = yaml.safe_load((sub_dir / 'eval_config.yaml').read_text())['evaluation']
    limits, gen_constraints = build_submission_limits()
    # Starter-notebook / HARNESS_README (2026-10-13) settings: compaction every 15 events.
    return EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=results_dir,
        submission_dir=sub_dir,
        models=models,
        sandbox='subprocess',
        timeout_seconds=int(ev.get('timeout_seconds', 300)),
        verification_timeout_seconds=int(ev.get('verification_timeout_seconds', 300)),
        max_time_minutes=float(ev.get('max_time_minutes', 60.0)),
        max_tool_calls=int(ev.get('max_tool_calls', 100)),
        max_turns=int(ev['max_turns']) if ev.get('max_turns') is not None else None,
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5,
        ),
        graph_dir=GRAPH_DIR,
        embeddings_dir=EMBEDDINGS_DIR,
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    )


def run_tasks(evaluator, label: str) -> list[dict]:
    rows = []
    CALIB.setdefault('runs', {})[label] = rows
    for idx, task in enumerate(SELECTED, start=1):
        t0 = time.time()
        try:
            r = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SELECTED))
            row = {'instance_id': task.instance_id, 'resolved': bool(r.resolved),
                   'wall_seconds': round(time.time() - t0, 1),
                   'duration_seconds': round(float(getattr(r, 'duration_seconds', 0) or 0), 1),
                   'tool_calls': getattr(r, 'tool_calls', None), 'llm_calls': getattr(r, 'total_llm_calls', None),
                   'patch_chars': len(getattr(r, 'agent_patch', '') or ''), 'error': (getattr(r, 'error', '') or '')[:200],
                   'test_exit_code': getattr(r, 'test_exit_code', None)}
            if not row['resolved']:
                row['test_output_tail'] = (getattr(r, 'test_output', '') or '')[-1500:]
        except Exception as e:
            row = {'instance_id': task.instance_id, 'resolved': False, 'wall_seconds': round(time.time() - t0, 1),
                   'exception': repr(e)[:300]}
        rows.append(row)
        save()
        print(label, idx, row)
    return rows

In [ ]:
# 1. Gold-patch control: does Kaggle grade these tasks? (no model needed)
from adk_submission import discover_adapters
from swegemma.cli import _build_cli_model_registry
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

t0 = time.time()
gold_models = _build_cli_model_registry(SUBS['31b'])
gold_adapters = discover_adapters(str(SUBS['31b']), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gold_eval = GoldEvaluator(make_eval_config(SUBS['31b'], gold_models, gold_adapters, WORKING_DIR / 'results_gold'))
gold_rows = run_tasks(gold_eval, 'gold')
CALIB['gold_resolved'] = sum(r['resolved'] for r in gold_rows)
CALIB['gold_seconds'] = round(time.time() - t0, 1)
save()
print('gold control:', CALIB['gold_resolved'], '/', len(gold_rows))

In [ ]:
# 2-3. For each model: start the server exactly as the starter does, probe speed, run the agent.
import urllib.request

import litellm
import torch
from adk_submission import VllmConfig, VllmServer
from adk_submission.discovery import discover_declared_models

litellm.drop_params = True
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

"""Measure raw serving speed of an OpenAI-compatible endpoint (stdlib only).

For several prompt sizes it reports prefill time (uncached and prefix-cached) and
decode tokens/second, plus one tool-call round trip. The same code is embedded in the
Kaggle timing notebook so local and L4x4 numbers are directly comparable.
"""


import json
import sys
import time
import urllib.request
import uuid

MODEL = 'gemma-4-31b-it-qat-w4a16-ct'

FILLER = (
    'def resolve_dependency(graph, node, seen=None):\n'
    '    """Walk the dependency graph depth-first and return nodes in install order."""\n'
    '    seen = set() if seen is None else seen\n'
    '    order = []\n'
    '    for child in sorted(graph.get(node, ())):\n'
    '        if child not in seen:\n'
    '            seen.add(child)\n'
    '            order.extend(resolve_dependency(graph, child, seen))\n'
    '    order.append(node)\n'
    '    return order\n\n'
)


def post(base: str, path: str, body: dict, timeout: float = 900.0) -> tuple[dict, float]:
    req = urllib.request.Request(
        base.rstrip('/') + path,
        data=json.dumps(body).encode(),
        headers={'Content-Type': 'application/json', 'Authorization': 'Bearer EMPTY'},
    )
    t0 = time.perf_counter()
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        out = json.loads(resp.read())
    return out, time.perf_counter() - t0


def chat(base: str, model: str, prompt: str, max_tokens: int, thinking: bool = False) -> tuple[dict, float]:
    body = {
        'model': model,
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
        'min_tokens': max_tokens,
        'temperature': 0.0,
        'chat_template_kwargs': {'enable_thinking': thinking},
    }
    return post(base, '/chat/completions', body)


def probe(base: str = 'http://127.0.0.1:8000/v1', model: str = MODEL, decode_tokens: int = 384) -> dict:
    results: dict = {'model': model, 'sizes': []}
    chat(base, model, 'Say hi.', 8)  # warm up

    for repeats in (8, 60, 120):  # roughly 1k / 8k / 16k prompt tokens
        nonce = uuid.uuid4().hex
        prompt = f'# session {nonce}\n' + FILLER * repeats + '\nExplain this code in great detail.'
        out1, t_prefill = chat(base, model, prompt, 1)
        out_cached, t_cached = chat(base, model, prompt, 1)
        out2, t_full = chat(base, model, prompt, decode_tokens)
        usage = out2.get('usage', {})
        completion = usage.get('completion_tokens', decode_tokens)
        decode_s = max(t_full - t_cached, 1e-6)
        results['sizes'].append({
            'prompt_tokens': usage.get('prompt_tokens'),
            'prefill_seconds_uncached': round(t_prefill, 2),
            'prefill_tok_per_s_uncached': round((usage.get('prompt_tokens') or 0) / max(t_prefill, 1e-6), 1),
            'prefill_seconds_cached': round(t_cached, 2),
            'completion_tokens': completion,
            'decode_tok_per_s': round((completion - 1) / decode_s, 2),
        })

    # One realistic agent-style turn: thinking on, a tool available.
    tools = [{
        'type': 'function',
        'function': {
            'name': 'read_file',
            'description': 'Read a file from the workspace.',
            'parameters': {
                'type': 'object',
                'properties': {'filepath': {'type': 'string'}},
                'required': ['filepath'],
            },
        },
    }]
    body = {
        'model': model,
        'messages': [{'role': 'user', 'content': 'Open src/app.py so we can find the bug in the login handler.'}],
        'tools': tools,
        'tool_choice': 'auto',
        'max_tokens': 1024,
        'temperature': 0.0,
    }
    out, t = post(base, '/chat/completions', body)
    msg = out['choices'][0]['message']
    results['tool_turn'] = {
        'seconds': round(t, 2),
        'completion_tokens': out.get('usage', {}).get('completion_tokens'),
        'has_reasoning': bool(msg.get('reasoning_content') or msg.get('reasoning')),
        'tool_calls': [c['function'] for c in (msg.get('tool_calls') or [])],
        'finish_reason': out['choices'][0].get('finish_reason'),
    }
    return results




def run_model(tag: str) -> None:
    sub_dir = SUBS[tag]
    declared = next(iter(discover_declared_models(sub_dir)))
    model_path = Path(f'/kaggle/input/models/google/gemma-4/other/{declared}/2')
    adapters = discover_adapters(str(sub_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    adapter_map = getattr(adapters, 'adapters', None) or {}
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
    tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    cfg = VllmConfig(
        model=str(model_path), port=8000, host='127.0.0.1',
        tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
        dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
        gpu_memory_utilization=0.90, enable_auto_tool_choice=True,
        enable_lora=bool(adapter_map), max_loras=max(1, len(adapter_map)), max_lora_rank=128,
        scheduler_reserve_full_isl=False, tensor_parallel_size=tp_size, startup_timeout=60 * 20,
        extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared, ()),
    )
    info = CALIB['models'].setdefault(tag, {'declared': declared})
    server = VllmServer(cfg, adapter_manifest=adapters)
    t0 = time.time()
    server.start()
    info['vllm_startup_seconds'] = round(time.time() - t0, 1)
    save()
    try:
        try:
            with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=30) as resp:
                served_ids = [m['id'] for m in json.loads(resp.read())['data']]
            info['served_model_ids'] = served_ids
            base_ids = [m for m in served_ids if m not in set(adapter_map)]
            info['speed'] = probe(model=base_ids[0])
        except Exception as e:  # the probe is a side measurement; never let it stop the agent runs
            info['speed_error'] = repr(e)[:300]
        save()
        models = server.create_model_registry(aliases=[declared], model_prefix='openai/', api_key=INFERENCE_API_KEY)
        evaluator = Evaluator(make_eval_config(sub_dir, models, adapters, WORKING_DIR / f'results_{tag}'))
        t0 = time.time()
        rows = run_tasks(evaluator, tag)
        info['agent_seconds'] = round(time.time() - t0, 1)
        info['resolved'] = sum(r['resolved'] for r in rows)
        gradable = {g['instance_id'] for g in CALIB['runs'].get('gold', []) if g['resolved']}
        info['resolved_on_gradable'] = sum(r['resolved'] for r in rows if r['instance_id'] in gradable)
        info['gradable'] = len(gradable)
        save()
    finally:
        server.stop()
        time.sleep(20)


for tag in ('31b', '12b'):
    try:
        run_model(tag)
    except Exception as e:
        CALIB['models'].setdefault(tag, {})['exception'] = repr(e)[:500]
        save()
        print(tag, 'failed:', e)
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != 'speed'} for k, v in CALIB['models'].items()}, indent=2))